In [0]:
-- 00_setup.sql
CREATE CATALOG IF NOT EXISTS rearc_quest
COMMENT 'Rearc Data Quest: BLS productivity + US population';
CREATE SCHEMA IF NOT EXISTS rearc_quest.raw COMMENT 'Landing volume and ingestion manifest';
CREATE SCHEMA IF NOT EXISTS rearc_quest.bronze COMMENT 'Raw rows as received, one table per
source file';
CREATE SCHEMA IF NOT EXISTS rearc_quest.silver COMMENT 'Typed, trimmed, de-duplicated,
conformed';
CREATE SCHEMA IF NOT EXISTS rearc_quest.gold COMMENT 'Stakeholder-facing answer tables';

CREATE VOLUME IF NOT EXISTS rearc_quest.raw.landing
COMMENT 'Byte-identical copies of source files. Never edited by hand.';
-- Manifest: one row per (source, file). Drives idempotent ingestion.
CREATE TABLE IF NOT EXISTS rearc_quest.raw.ingest_manifest (
source STRING NOT NULL COMMENT 'bls_pr | datausa_population',
file_name STRING NOT NULL COMMENT 'Remote file name or logical name for API
payloads',
volume_path STRING COMMENT 'Where the current copy lives in the Volume',
remote_last_modified TIMESTAMP COMMENT 'Last-Modified from directory listing / HTTP
header',
remote_size_bytes BIGINT COMMENT 'Content-Length / listing size',
sha256 STRING COMMENT 'Hash of the landed bytes',
status STRING NOT NULL COMMENT 'active | removed',
first_seen_at TIMESTAMP NOT NULL,
last_seen_at TIMESTAMP NOT NULL COMMENT 'Last time the remote listing contained this
file',
last_changed_at TIMESTAMP COMMENT 'Last time the landed bytes changed',
CONSTRAINT pk_manifest PRIMARY KEY (source, file_name)
) COMMENT '
Ingestion ledger. Re-runs compare the remote listing against this table.';
-- Quick check
LIST '/Volumes/rearc_quest/raw/landing/';